**Code to open/merge data files on pressure levels (3D winds).**

Code generated by John Cramblitt and John Chiang.

Please contact John Cramblittt (jcramblitt@berkeley.edu) with questions or concerns.

In [10]:
from netCDF4 import Dataset
import dask
import numpy as np
import numpy.ma as ma
import matplotlib 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import mpl_toolkits
import glob
import pandas as pd
import xarray as xr
import scipy as sp
from scipy import signal
import scipy.stats
import scipy.integrate
import os
import windspharm
import spharm
import _spherepack 
import cftime
from dask.diagnostics import ProgressBar
from pathlib import Path
import re
import gc

In [3]:
## open/merge data files using metadata in file name ##

BASE = Path(r"atm_pressure_levels")
PREFIX = "atm_monthly_script_"
SUFFIX = ".nc"
CHUNKS = {"time": 120} 
ENGINE = None                         

rx = re.compile(rf"^{re.escape(PREFIX)}"r"(?:(?P<location>.+?)_VolcanoStart|(?P<no_volcano>NoVolcano)Start)"r"(?P<stamp>\d{8})"r"(?:-e(?P<ensemble>\d+))?"rf"{re.escape(SUFFIX)}$")

def scenario_label(stamp):
    return stamp[-4:]

files_by_loc = {}
for p in BASE.glob(f"{PREFIX}*{SUFFIX}"):
    m = rx.match(p.name)
    if not m:
        continue
    # if it's a NoVolcano file, location="NoVolcano"
    loc = m.group("location") or m.group("no_volcano")
    scen = scenario_label(m.group("stamp"))
    ens  = f"e{m.group('ensemble')}" if m.group("ensemble") else "e0"

    files_by_loc.setdefault(loc, {}).setdefault(scen, {})[ens] = p

SCEN_ORDER = ["1923", "1927"]

def open_location_block(loc, scen_map, chunks=CHUNKS, engine=ENGINE):
    scen_labels = [s for s in SCEN_ORDER if s in scen_map] + [s for s in sorted(scen_map) if s not in SCEN_ORDER]

    scenario_dsets = []
    for scen in scen_labels:
        ens_map = scen_map[scen]
        ens_labels = sorted(ens_map) 
        paths = [ens_map[e] for e in ens_labels]

        ds_ens = xr.open_mfdataset(paths,
                                   combine="nested",
                                   concat_dim="ensemble",
                                   chunks=chunks,
                                   engine=engine,
                                   parallel=True,
                                   decode_times=True,
                                   use_cftime="auto")
        ds_ens = ds_ens.assign_coords(ensemble=ens_labels)
        scenario_dsets.append(ds_ens)

    ds = xr.concat(scenario_dsets, dim="scenario")
    ds = ds.assign_coords(scenario=scen_labels).expand_dims(location=[loc])
    return ds

blocks = [open_location_block(loc, scen_map) for loc, scen_map in sorted(files_by_loc.items())]
ds_all = xr.concat(blocks, dim="location")

# ds_all = ds_all.squeeze("lev", drop=True) # drop unnecessary coord
# convert from -180-180 to 0-360
ds_all = ds_all.assign_coords(lon=(((ds_all.lon + 180) % 360) - 180)).sortby("lon")

In [ ]:
 # keep only variables of interest, and first 12 months to limit dataset size
vars = ds_all[['U','V','OMEGA']].squeeze('scenario').isel(time=slice(0,12)).chunk('auto')

gc.collect() # help with memory

vars.sel(scenario='1923').to_netcdf("Pausata_all_atm_1923.nc")
vars.sel(scenario='1927').to_netcdf("Pausata_all_atm_1927.nc")